<h1 style=\"text-align: center; font-size: 50px;\"> <h1 style=\"text-align: center; font-size: 50px;\"> 📦 Register Model </h1> </h1>

📘 Project Overview: 
 This notebook demonstrates a modular architecture for answering natural language questions 
 over one or more feedback documents using only local and open-source models (e.g., LLaMA.cpp).
 The system processes long documents chunk-by-chunk and synthesizes a final answer using a multi-step LLM workflow.

# Notebook Overview

- Start Execution
- Define User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- KV Memory
- LLM Setup
- State Model
- Node Functions
- Graph Definition
- Graph Visualization
- Generated Answer
- Message History

# Start Execution

In [ ]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations
from pathlib import Path  # Object-oriented file system paths

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.model_selection import ModelSelector
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state
from src.utils import (  # Utility functions for logging, LLM I/O, and schema generation
    load_config,
    load_secrets,
    load_secrets_to_env,
    configure_proxy,
    display_image,
    get_project_root,
    get_response_from_llm,
    json_schema_from_type,
    log_timing,
    sec_to_timestamp,
    logger,
    login_huggingface,
    setup_model_environment,
    ensure_wav,
)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Define User Constants

In [ ]:
# TOPIC: str = "Focus Flow"  
# QUESTION: str = "Which poeple provided the feedback?"

QUESTION: str = "What are the main issues regarding the product"
DOCS: list
FILE_ID: str
MEMORY: SimpleKVMemory
INPUT_PATH: Path = Path("../data/input")

# Install and Import Libraries

In [4]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 18.3 ms, sys: 0 ns, total: 18.3 ms
Wall time: 991 ms


In [ ]:

# from src.utils import logger, ensure_wav, setup_model_environment, format_model_path

from __future__ import annotations  # Enables postponed evaluation of annotations (PEP 563)
# import soundfile  # Library for reading and writing sound files

# ─────── Standard Library ───────
import base64  # Provides encoding and decoding of binary data
import functools  # Higher-order functions for functional programming 
import json  # JSON serialization and deserialization
import logging  # Flexible logging system
import multiprocessing  # Support for spawning processes
import shutil  # High-level file operations
import warnings  # Issue warning messages
from collections import namedtuple  # Factory for creating tuple subclasses with named fields
from datetime import datetime  # Date and time utilities
from pathlib import Path  # Object-oriented filesystem paths
from typing import Any, Dict, List, Literal, Optional, TypedDict  # Type hinting support
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq


# ─────── Third-Party Packages ───────
import mlflow  # Model tracking and serving framework
from mlflow.tracking import MlflowClient  # Interface to interact with MLflow tracking server for experiments, runs, and artifacts
import yaml  # YAML parsing and serialization
from IPython.display import Markdown, display  # IPython utilities for notebook output formatting
from tqdm import tqdm  # Visual progress bar for loops
from huggingface_hub import snapshot_download, hf_hub_download
import torch

# ─────── LangChain Core & Community ───────
from langchain.docstore.document import Document  # Core document abstraction
from langchain_community.llms import LlamaCpp  # Integration for local LlamaCpp models

from src.agentic_workflow import build_agentic_graph
from src.agentic_audio_rag_model import AgenticAudioRAGModel  # Core agent logic for audio RAG tasks
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state

# Configure Settings

In [6]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [ ]:
project_root = get_project_root()
INPUT_PATH: Path = Path("../data/input")  

MEMORY_PATH: Path = Path("../data/memory")
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"

LLAMA_MODEL_PATH = "/home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
SAMPLE_MEDIA_PATH = INPUT_PATH / "sample_tts.mp3"
CONTEXT_WINDOW = 8192
MAX_TOKENS = CONTEXT_WINDOW // 8
CHUNK_SIZE = CONTEXT_WINDOW // 2
CHUNK_OVERLAP = CHUNK_SIZE // 8  

EXPERIMENT_NAME = "AIStudio-Agentic-Audio-RAG-with-LangGraph-Experiment"
RUN_NAME = "AIStudio-Agentic-Audio-RAG-with-LangGraph-Run"
MODEL_NAME = "AIStudio-Agentic-Audio-RAG-with-LangGraph-Model"

In [ ]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

In [8]:
logger.info('Notebook execution started.')

## Verify Assets

In [ ]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

def log_secrets_status(secrets: Dict[str, Any], success_message: str, failure_message: str) -> None:
    """
    Logs the status of secrets based on their existence.

    Parameters:
        secrets (Dict[str, Any]): Secrets retrieved to check if they exist.
        success_message (str): Message to log if secrets exists.
        failure_message (str): Message to log if secrets do not exist.
    """
    if secrets:
        logger.info(f"Project secrets are available. {success_message}")
    else:
        logger.info(f"There are no project secrets found. {failure_message}")

In [ ]:
log_asset_status(
    asset_path=INPUT_PATH,
    asset_name="Input Data",
)

log_asset_status(
    asset_path=LLAMA_MODEL_PATH,
    asset_name="LLM",
)

log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="Config",
)

log_secrets_status(
    secrets=secrets,
    success_message="",
    failure_message="Please check if the secrets were propely connfigured in your secrets yaml file or in Secrets Manager."
)

# KV Memory

In [11]:
memory: SimpleKVMemory = SimpleKVMemory(MEMORY_PATH)
memory.set('dummy key', 'dummy value')

# Prepare Audio Files for Inference

In [ ]:
logger.info("🎧 Scanning directory for media files: %s", INPUT_PATH)

# Prefer an audio-native LLM
AUDIO_LLM = {
 #   "MiDaSheng": ("MiSpeech/MiDaShengLM-7B-GGUF"),
 #   "Kimi": ("Moonshot-AI/Kimi-Audio-7B-Instruct"),
    "Qwen": ("Qwen/Qwen2.5-Omni-7B"), 
}["Qwen"]

# Supported media types
AUDIO_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv"}
MEDIA_EXTS = AUDIO_EXTS | VIDEO_EXTS

# Ensure HF cache paths live in the project area (matches README/setup)
setup_model_environment()

In [ ]:

# def _ensure_hf_local(model_id: str) -> str:
#     """
#     Ensure the model exists locally; download snapshot if needed.
#     Uses project-local cache path derived by utils.format_model_path().
#     """
#     selector = ModelSelector()
#     local_dir = selector.format_model_path(model_id)
#     local_dir.mkdir(parents=True, exist_ok=True)
#     # Download only if the directory is empty
#     if not any(local_dir.iterdir()):
#         logger.info("⬇️ Downloading audio LLM '%s' to %s", model_id, str(local_dir))
#         snapshot_download(
#             repo_id=model_id,
#             local_dir=str(local_dir),
#             local_dir_use_symlinks=False,
#             resume_download=True,
#         )
#     return str(local_dir)

# def transcribe_with_audio_llm(media_path: str, model_id: str) -> Dict[str, Any]:
#     """
#     Transcribe an audio/video file using an audio-native LLM (no Whisper).
#     Returns {"text": <full transcript>, "segments": [{"start": float, "end": float, "text": str}, ...]}
#     """
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     local_dir = _ensure_hf_local(model_id)

#     # Load processor + model
#     processor = AutoProcessor.from_pretrained(local_dir, trust_remote_code=True)
#     model = AutoModelForSpeechSeq2Seq.from_pretrained(
#         local_dir,
#         torch_dtype=torch.float16 if device == "cuda" else torch.float32,
#         trust_remote_code=True,
#     ).to(device)

#     # Convert to wav (mono, 16 kHz) when needed
#     wav_path = ensure_wav(media_path)

#     # Try processor-native loader first, then soundfile
#     try:
#         audio, sr = processor.audio_load(wav_path)  # some processors expose this
#     except Exception:
#         audio, sr = soundfile.read(wav_path)

#     inputs = processor(audio=audio, sampling_rate=sr, return_tensors="pt").to(device)

#     with torch.no_grad():
#         generated = model.generate(**inputs, max_new_tokens=8192)

#     text = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

#     # If the model doesn’t return diarized/segmented timestamps,
#     # provide a single coarse segment as a placeholder.
#     # Downstream chunking will enrich this into overlapping windows.
#     est_seconds = max(10.0, len(text.split()) / 2.5)
#     segments = [{"start": 0.0, "end": float(est_seconds), "text": text}]

#     return {"text": text, "segments": segments}

# # -------- Scan & ingest media --------
# all_docs: List[Document] = []

# media_files = []
# for file_path in Path(INPUT_PATH).rglob("*"):
#     # Skip hidden/system folders
#     if any(part.startswith(".") and part not in {".", ".."} for part in file_path.parts):
#         continue
#     if file_path.suffix.lower() in MEDIA_EXTS:
#         media_files.append(file_path)

# if not media_files:
#     logger.warning("📭 No audio/video files found in %s", INPUT_PATH)

# for media_path in media_files:
#     try:
#         logger.info("🔊 Transcribing: %s  (model=%s)", media_path.name, AUDIO_LLM)
#         result = transcribe_with_audio_llm(str(media_path), AUDIO_LLM)

#         transcript_text = result["text"]
#         segments = result.get("segments", [])

#         # Wrap as a single LangChain Document (full transcript).
#         # Timestamps are preserved in metadata for downstream UI/reranker.
#         doc = Document(
#             page_content=transcript_text,
#             metadata={
#                 "file_path": str(media_path),
#                 "file_name": media_path.name,
#                 "media_type": "audio" if media_path.suffix.lower() in AUDIO_EXTS else "video",
#                 "segments": segments,  # [{"start": float, "end": float, "text": str}, ...]
#                 "source": "audio_llm_transcription",
#                 "audio_llm": AUDIO_LLM,
#             },
#         )
#         all_docs.append(doc)
#         logger.info("✅ Loaded transcript as Document: %s (chars=%d)", media_path.name, len(transcript_text))
#     except Exception as e:
#         logger.warning("❌ Failed to transcribe %s: %s", media_path.name, e)

# logger.info("📦 Total media files processed: %d — Documents created: %d", len(media_files), len(all_docs))


# INPUT_TEXT = '\n\n'.join([doc.page_content for doc in all_docs])

# MLflow Registration

In [14]:
# 1. Set MLflow tracking URI and experiment
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

Using MLflow tracking URI: /phoenix/mlflow
Experiment: AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Experiment


In [15]:
%%time

# These should point to the actual files you're using for model and memory
MODEL_ARTIFACTS = {
    "model_path": str(MODEL_PATH),
    "memory_path": str(MEMORY_PATH),
}
 
# === Start MLflow run, log, and register ===
with mlflow.start_run(run_name=RUN_NAME) as run:
    print(f"🚀 Started MLflow run: {run.info.run_id}")

    # Log and register the model using the classmethod
    AgenticFeedbackModel.log_model(
        model_name=MODEL_NAME,
        model_artifacts=MODEL_ARTIFACTS
    )

logger.info(f"✅ Model '{MODEL_NAME}' successfully logged and registered.")

2025/08/02 06:43:49 INFO mlflow.models.signature: Inferring model signature from type hints


🚀 Started MLflow run: e015ccbb3a024c3e9ab35a177ab9d238


Successfully registered model 'AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Model'.
Created version '1' of model 'AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Model'.


CPU times: user 1 s, sys: 15.7 s, total: 16.7 s
Wall time: 4min 3s


In [16]:
# 3. Retrieve the latest version from the Model Registry
client = MlflowClient()
versions = client.get_latest_versions(MODEL_NAME, stages=["None"])

if not versions:
    raise RuntimeError(f"No registered versions found for model '{MODEL_NAME}'.")
    
latest_version = versions[0].version
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_version}")

logger.info(f"Latest registered version of '{MODEL_NAME}': {latest_version}")
logger.info(f"Signature: {model_info.signature}")

In [17]:
%%time

# 4. Load the model from the Model Registry
loaded_model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_version}")
logger.info(f"Successfully loaded model '{MODEL_NAME}' version {latest_version} for inference.")

CPU times: user 1.22 s, sys: 2.34 s, total: 3.56 s
Wall time: 1min 13s


In [ ]:
# 5. Run a sample inference using the loaded model (Audio RAG)

from pathlib import Path

# Collect media files (same extensions you used in the ingestion cell)
_MEDIA_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a", ".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}
sample_media_paths = [
    str(p) for p in sorted(Path(INPUT_PATH).rglob("*"))
    if p.is_file() and p.suffix.lower() in _MEDIA_EXTS
]

if not sample_media_paths:
    raise FileNotFoundError(f"No audio/video files found in {INPUT_PATH}. "
                            f"Please add at least one media file to run a sample inference.")

# The audio RAG model expects {"paths": [...], "query": "..."}
input_payload = [{
    "paths": sample_media_paths,
    "query": QUESTION  # reuse your QUESTION var, or set a literal test query here
}]

print("\n=== Running Sample Inference (Audio RAG) ===")
results = loaded_model.predict(input_payload)       # preserve list-in / list-out contract
result = results[0] if isinstance(results, list) else results

print("Answer:\n", result["answer"])
print("\nSupporting chunks:")
for c in result.get("chunks", []):
    print(f"- [{c['start_s']:.1f}–{c['end_s']:.1f}s] {c['text'][:120]}...")




=== Running Sample Inference ===


🔁 Processing each chunk: 100%|██████████| 2/2 [00:01<00:00,  1.96it/s, group=✅ Chunk 2 response length: 28 chars]


🔁 Processing each grouped chunk answers: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it, group=🧠 Synthesized partial answer (1/1)]



🔚 === Final Answer ===

# 🧠 Synthesized partial answer (1/1)

Since the user's question is about individuals mentioned in the document as providing feedback, and neither Chunk 1 nor Chunk 2 mentions any individuals providing feedback, the final answer is:

**No individuals are mentioned in the document as providing feedback.**




# Generated Answer

In [19]:
display(Markdown(result.answer))

# 🧠 Synthesized partial answer (1/1)

Since the user's question is about individuals mentioned in the document as providing feedback, and neither Chunk 1 nor Chunk 2 mentions any individuals providing feedback, the final answer is:

**No individuals are mentioned in the document as providing feedback.**

# Message History

In [20]:
print(result.messages)

[
    {
        "role": "developer",
        "content": "User submitted a question."
    },
    {
        "role": "user",
        "content": "Which poeple provided the feedback?"
    },
    {
        "role": "developer",
        "content": "\ud83e\udde0 Relevance check result:"
    },
    {
        "role": "assistant",
        "content": "yes"
    },
    {
        "role": "developer",
        "content": "\ud83e\udded No cached answer found for question: 'Which poeple provided the feedback?'"
    },
    {
        "role": "developer",
        "content": "\u270f\ufe0f Rewritten user question:"
    },
    {
        "role": "assistant",
        "content": "Who are the individuals mentioned in the document as providing feedback?"
    },
    {
        "role": "developer",
        "content": "\ud83e\udde9 Chunked 1 documents into 2 chunks (size=4096, overlap=256)"
    },
    {
        "role": "developer",
        "content": "\ud83e\udde0 Processed 2 chunks for question: 'Who are the individual

In [21]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).